# MindScope: Fine-Tuning DistilBERT for Youth Emotion Classification (Colab GPU)

**Project:** MindScope - AI-Based Analysis of Causes and Effects of Youth Mental Health Issues  
**Target Architecture:** `distilbert-base-uncased` fine-tuned on emotion annotations (GoEmotions / Student Reflections).  
**Execution Environment:** Google Colab / Kaggle with T4 / P100 GPU.  
**Deployment Target:** CPU Inference in FastAPI backend.  

> **Ethical & Safety Notice:** MindScope is strictly a research and educational decision-support tool, NOT a diagnostic instrument. All text processing is coupled with deterministic crisis safeguards.

In [ ]:
# 1. Install necessary dependencies
!pip install -q transformers datasets accelerate evaluate scikit-learn huggingface_hub

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Training Device: {device}")

## 2. Dataset Loading & Preprocessing
We map fine-grained affective labels to 6 core youth emotional categories:
0. `sadness`
1. `anxiety_fear`
2. `anger`
3. `stress_overwhelm`
4. `joy_optimism`
5. `neutral`

In [ ]:
MODEL_CKPT = "distilbert-base-uncased"
NUM_LABELS = 6
LABEL_NAMES = ["sadness", "anxiety_fear", "anger", "stress_overwhelm", "joy_optimism", "neutral"]
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

# Load GoEmotions dataset
dataset = load_dataset("go_emotions", "simplified")

def map_labels(batch):
    # Simplified mapping to primary affective dimensions
    mapped_labels = []
    for labels in batch["labels"]:
        if not labels:
            mapped_labels.append(5)  # neutral
        elif any(l in [25, 15] for l in labels):  # sadness, grief
            mapped_labels.append(0)
        elif any(l in [14, 19] for l in labels):  # fear, nervousness
            mapped_labels.append(1)
        elif any(l in [2, 3] for l in labels):   # anger, annoyance
            mapped_labels.append(2)
        elif any(l in [10, 11] for l in labels):  # disappointment, embarrassment / stress
            mapped_labels.append(3)
        elif any(l in [17, 20, 22] for l in labels):  # joy, optimism, pride
            mapped_labels.append(4)
        else:
            mapped_labels.append(5)
    return {"label": mapped_labels}

dataset = dataset.map(map_labels, batched=True)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

## 3. Metrics Definition & Trainer Setup

In [ ]:
f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    accuracy = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"macro_f1": macro_f1, "accuracy": accuracy}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="./distilbert_emotion_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Start Fine-Tuning
trainer.train()

## 4. Model Export & Optimization for CPU Deployment
Save the lightweight model weights and tokenizer for CPU inference.

In [ ]:
# Export artifacts
SAVE_DIR = "./mindscope_distilbert_emotion"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Successfully saved fine-tuned model to {SAVE_DIR}")

# Verify CPU Inference
from transformers import pipeline
cpu_clf = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR, device=-1)
print(cpu_clf("I have so many project deadlines and can barely sleep."))